### Financial Inclusion — Feature Engineering & Encoding

---

### Encoding strategy overview

| Column | Action | Method | Reason |
|---|---|---|---|
| `year` | ❌ Drop | — | No predictive signal; all data is 2016–2018 |
| `uniqueid` | ❌ Drop | — | Identifier, not a feature |
| `household_size` | ❌ Drop | — | Weak proxy; correlated with other features |
| `relationship_with_head` | ❌ Drop | — | Too granular; correlates with gender/age |
| `marital_status` | ❌ Drop | — | Many 'Dont know' values; limited signal |
| `country` | ✅ Keep | One-Hot Encoding | Nominal — no natural order across 4 countries |
| `location_type` | ✅ Keep | Binary (0/1) | Only 2 values: Rural=0, Urban=1 |
| `cellphone_access` | ✅ Keep | Binary (0/1) | Only 2 values: No=0, Yes=1 |
| `gender_of_respondent` | ✅ Keep | Binary (0/1) | Only 2 values: Female=0, Male=1 |
| `age_of_respondent` | ✅ Keep | Keep as-is (numeric) | Already numeric and continuous |
| `education_level` | ✅ Keep | Ordinal Encoding | Clear progression from no education → tertiary |
| `job_type` | ✅ Keep | One-Hot Encoding | No meaningful order across 10 job categories |

---

### Step 0 — Imports

In [21]:
import pandas as pd
import numpy as np

---
### Step 1 — Load the Raw Data

In [22]:
df = pd.read_csv('data/Test.csv')

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 10,086 rows × 12 columns


,country,year,uniqueid,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_6056,Urban,Yes,3,30,Male,Head of Household,Married/Living together,Secondary education,Formally employed Government
1,Kenya,2018,uniqueid_6060,Urban,Yes,7,51,Male,Head of Household,Married/Living together,Vocational/Specialised training,Formally employed Private
2,Kenya,2018,uniqueid_6065,Rural,No,3,77,Female,Parent,Married/Living together,No formal education,Remittance Dependent


---
### Step 2 — Drop Columns

**Dropped:**
- `year` — all surveys were 2016–2018; no meaningful signal
- `uniqueid` — row identifier, not a feature
- `household_size` — weak proxy; adds noise without clear signal
- `relationship_with_head` — overlaps with age/gender; too granular
- `marital_status` — many 'Dont know' values; limited ML signal

In [23]:
COLS_TO_DROP = [
    'year',
    'uniqueid',
    'household_size',
    'relationship_with_head',
    'marital_status',
]

df = df.drop(columns=COLS_TO_DROP)

print(f'Remaining columns ({len(df.columns)}):')
for col in df.columns:
    print(f'  • {col}')

Remaining columns (7):
  • country
  • location_type
  • cellphone_access
  • age_of_respondent
  • gender_of_respondent
  • education_level
  • job_type


---
### Step 3 — Binary Encoding

Columns with exactly **2 categories** → converted to 0 or 1.

| Column | 0 | 1 |
|---|---|---|
| `location_type` | Rural | Urban |
| `cellphone_access` | No | Yes |
| `gender_of_respondent` | Female | Male |

In [24]:
# location_type: Rural=0, Urban=1
df['bin_location_type'] = df['location_type'].map({'Rural': 0, 'Urban': 1})

# cellphone_access: No=0, Yes=1
df['bin_cellphone_access'] = df['cellphone_access'].map({'No': 0, 'Yes': 1})

# gender_of_respondent: Female=0, Male=1
df['bin_gender'] = df['gender_of_respondent'].map({'Female': 0, 'Male': 1})

# Verify — no NaN should appear if all values are mapped
checks = ['bin_location_type', 'bin_cellphone_access', 'bin_gender']
for col in checks:
    nulls = df[col].isnull().sum()
    vals = df[col].value_counts().to_dict()
    print(f'{col}: {vals}  |  NaN: {nulls}')

bin_location_type: {0: 6189, 1: 3897}  |  NaN: 0
bin_cellphone_access: {1: 7559, 0: 2527}  |  NaN: 0
bin_gender: {0: 5847, 1: 4239}  |  NaN: 0


---
### Step 4 — Ordinal Encoding: Education Level

Education has a **clear natural order** from lowest to highest — so we assign a number that respects that order.

| Level | Encoded value |
|---|---|
| No formal education | 0 |
| Primary education | 1 |
| Secondary education | 2 |
| Vocational/Specialised training | 3 |
| Tertiary education | 4 |
| Other/Dont know/RTA | 0 *(treated as unknown = lowest)* |


In [25]:
education_order = {
    'No formal education': 0,
    'Primary education': 1,
    'Secondary education': 2,
    'Vocational/Specialised training': 3,
    'Tertiary education': 4,
    'Other/Dont know/RTA': 0,
}

df['ord_education_level'] = df['education_level'].map(education_order)

print('Education encoding check:')
check = df.groupby('education_level')['ord_education_level'].first().reset_index()
check.columns = ['Original label', 'Encoded value']
check = check.sort_values('Encoded value')
print(check.to_string(index=False))
print(f'\nNaN count: {df["ord_education_level"].isnull().sum()}')

Education encoding check:
                 Original label  Encoded value
            No formal education              0
            Other/Dont know/RTA              0
              Primary education              1
            Secondary education              2
Vocational/Specialised training              3
             Tertiary education              4

NaN count: 0


---
### Step 5 — One-Hot Encoding: Country & Job Type

**One-Hot encoding** creates a separate 0/1 column for each category. This is the right choice when:
- There is **no natural order** between categories
- The model should treat each category independently

We drop the first dummy column (`drop_first=True`) to avoid the *dummy variable trap* (multicollinearity).

With 4 countries, this creates 3 new columns (Kenya as baseline). With 10 job types, this creates 9 new columns.

In [26]:
# --- Country ---
country_dummies = pd.get_dummies(
    df['country'],
    prefix='ohe_country',
    drop_first=True   # Kenya is the baseline (dropped)
).astype(int)

print('Country dummy columns created:')
for col in country_dummies.columns:
    print(f'  • {col}  (1 = yes, 0 = no)')
print('  → Baseline (dropped): Kenya')

# --- Job Type ---
job_dummies = pd.get_dummies(
    df['job_type'],
    prefix='ohe_job',
    drop_first=True   # "Dont Know/Refuse to answer" is the baseline (dropped)
).astype(int)

print('\nJob type dummy columns created:')
for col in job_dummies.columns:
    print(f'  • {col}')

# Attach to main dataframe
df = pd.concat([df, country_dummies, job_dummies], axis=1)

print(f'\nDataFrame shape after adding dummies: {df.shape}')

Country dummy columns created:
  • ohe_country_Rwanda  (1 = yes, 0 = no)
  • ohe_country_Tanzania  (1 = yes, 0 = no)
  • ohe_country_Uganda  (1 = yes, 0 = no)
  → Baseline (dropped): Kenya

Job type dummy columns created:
  • ohe_job_Farming and Fishing
  • ohe_job_Formally employed Government
  • ohe_job_Formally employed Private
  • ohe_job_Government Dependent
  • ohe_job_Informally employed
  • ohe_job_No Income
  • ohe_job_Other Income
  • ohe_job_Remittance Dependent
  • ohe_job_Self employed

DataFrame shape after adding dummies: (10086, 23)


---
### Step 6 — Build the Final Clean Feature Table

Now we assemble the final table by selecting only the **encoded columns** and the **numeric columns**. We drop the original raw text columns.

In [27]:
# Columns to keep in the final ML-ready table
# ─ Numeric (kept as-is)
numeric_cols = ['age_of_respondent']

# ─ Binary encoded
binary_cols = ['bin_location_type', 'bin_cellphone_access', 'bin_gender']

# ─ Ordinal encoded
ordinal_cols = ['ord_education_level']

# ─ One-hot encoded (country + job)
ohe_cols = [c for c in df.columns if c.startswith('ohe_')]

final_cols = numeric_cols + binary_cols + ordinal_cols + ohe_cols

df_model = df[final_cols].copy()

print(f'Final feature table: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns')
print(f'\nColumn list ({len(df_model.columns)} total):')
for col in df_model.columns:
    dtype = str(df_model[col].dtype)
    print(f'  {col:<45} [{dtype}]')

Final feature table: 10,086 rows × 17 columns

Column list (17 total):
  age_of_respondent                             [int64]
  bin_location_type                             [int64]
  bin_cellphone_access                          [int64]
  bin_gender                                    [int64]
  ord_education_level                           [int64]
  ohe_country_Rwanda                            [int64]
  ohe_country_Tanzania                          [int64]
  ohe_country_Uganda                            [int64]
  ohe_job_Farming and Fishing                   [int64]
  ohe_job_Formally employed Government          [int64]
  ohe_job_Formally employed Private             [int64]
  ohe_job_Government Dependent                  [int64]
  ohe_job_Informally employed                   [int64]
  ohe_job_No Income                             [int64]
  ohe_job_Other Income                          [int64]
  ohe_job_Remittance Dependent                  [int64]
  ohe_job_Self employed          

---
### Step 7 — Final Checks

In [28]:
print('=== Final data quality check ===')
print(f'Shape         : {df_model.shape}')
print(f'Missing values: {df_model.isnull().sum().sum()}')
print(f'Data types    : {df_model.dtypes.value_counts().to_dict()}')
print()
print('Value ranges for numeric/ordinal columns:')
print(df_model[numeric_cols + ordinal_cols].describe().round(2))

=== Final data quality check ===
Shape         : (10086, 17)
Missing values: 0
Data types    : {dtype('int64'): 17}

Value ranges for numeric/ordinal columns:
       age_of_respondent  ord_education_level
count           10086.00             10086.00
mean               38.31                 1.22
std                16.27                 0.95
min                16.00                 0.00
25%                26.00                 1.00
50%                35.00                 1.00
75%                48.00                 2.00
max               100.00                 4.00


In [29]:
# Preview the first 5 rows of the final table
print('First 5 rows of the model-ready feature table:')
df_model.head()

First 5 rows of the model-ready feature table:


,age_of_respondent,bin_location_type,bin_cellphone_access,bin_gender,ord_education_level,ohe_country_Rwanda,ohe_country_Tanzania,ohe_country_Uganda,ohe_job_Farming and Fishing,ohe_job_Formally employed Government,ohe_job_Formally employed Private,ohe_job_Government Dependent,ohe_job_Informally employed,ohe_job_No Income,ohe_job_Other Income,ohe_job_Remittance Dependent,ohe_job_Self employed
0,30,1,1,1,2,0,0,0,0,1,0,0,0,0,0,0,0
1,51,1,1,1,3,0,0,0,0,0,1,0,0,0,0,0,0
2,77,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,39,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16,1,0,1,2,0,0,0,0,0,0,0,0,0,0,1,0


---
### Step 8 — Save the Feature Table

In [30]:
output_path = 'data/features_encoded_TEST.csv'
df_model.to_csv(output_path, index=False)

print(f'✅ Saved to: {output_path}')
print(f'   Rows   : {df_model.shape[0]:,}')
print(f'   Columns: {df_model.shape[1]}')

✅ Saved to: features_encoded.csv
   Rows   : 10,086
   Columns: 17


---
### Encoding Summary & Column Reference

| Final column | Original column | Encoding | Notes |
|---|---|---|---|
| `age_of_respondent` | `age_of_respondent` | Numeric | No change needed |
| `bin_location_type` | `location_type` | Binary | Rural=0, Urban=1 |
| `bin_cellphone_access` | `cellphone_access` | Binary | No=0, Yes=1 |
| `bin_gender` | `gender_of_respondent` | Binary | Female=0, Male=1 |
| `ord_education_level` | `education_level` | Ordinal 0–4 | Respects education hierarchy |
| `ohe_country_Rwanda` | `country` | One-Hot | 1 if Rwanda, else 0 |
| `ohe_country_Tanzania` | `country` | One-Hot | 1 if Tanzania, else 0 |
| `ohe_country_Uganda` | `country` | One-Hot | 1 if Uganda, else 0 |
| `ohe_job_Farming and Fishing` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Formally employed Government` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Formally employed Private` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Government Dependent` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Informally employed` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_No Income` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Other Income` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Remittance Dependent` | `job_type` | One-Hot | 1 if this job, else 0 |
| `ohe_job_Self employed` | `job_type` | One-Hot | 1 if this job, else 0 |

**Baseline categories (dropped to avoid multicollinearity):**
- Country baseline: `Kenya`
- Job baseline: `Dont Know/Refuse to answer`

---

### Next Step: Model Training

The `features_encoded.csv` file is now ready. When you load the **training data** (which includes the `bank_account` target column), apply the same encoding steps. Then you can train a model:

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = df_model  # features
y = train['bank_account'].map({'Yes': 1, 'No': 0})  # target (from training set)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
print(f'ROC-AUC: {roc_auc_score(y_val, y_pred):.4f}')
```